In [1]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import WebKB
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = WebKB(root=root, name=dataset_name)
    elif split_type == "60:20:20":
        transform = RandomNodeSplit(split="train_rest", num_val=0.2, num_test=0.2)
        dataset = WebKB(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))
        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "Cornell"
SPLIT_TYPE = "60:20:20"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[I 2026-09-22 10:17:43,341] A new study created in memory with name: no-name-6cc3cdd8-3305-4556-be95-483eeffa3cf9



Optuna search for GCN


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


...

[I 2026-09-22 10:22:23,145] Trial 0 finished with value: 0.5945946176846822 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.5945946176846822.
[I 2026-09-22 10:22:31,657] Trial 1 finished with value: 0.5945945978164673 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 0 with value: 0.5945946176846822.
[I 2026-09-22 10:22:34,253] Trial 2 finished with value: 0.6306306521097819 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 2 with value: 0.6306306521097819.
[I 2026-09-22 10:22:38,015] Trial 3 finished with value: 0.5855855941772461 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.6306

[I 2026-09-22 10:25:57,886] A new study created in memory with name: no-name-e383fbe8-ea6e-4d81-aaf9-ae1225f6540c


GCNII: 0.5622 +/- 0.0602

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-22 10:25:59,155] Trial 0 finished with value: 0.7747748096783956 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.7747748096783956.
[I 2026-09-22 10:26:01,874] Trial 1 finished with value: 0.792792816956838 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 1 with value: 0.792792816956838.
[I 2026-09-22 10:26:03,389] Trial 2 finished with value: 0.7837838133176168 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 1 with value: 0.792792816956838.
[I 2026-09-22 10:26:05,689] Trial 3 finished with value: 0.8198198477427164 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 3 with value: 0.819819847742716

In [2]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import WebKB
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = WebKB(root=root, name=dataset_name)
    elif split_type == "60:20:20":
        transform = RandomNodeSplit(split="train_rest", num_val=0.2, num_test=0.2)
        dataset = WebKB(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))
        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "Texas"
SPLIT_TYPE = "60:20:20"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[I 2026-09-22 10:27:12,662] A new study created in memory with name: no-name-a50932da-160e-4567-b841-e93e025191da



Optuna search for GCN


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


...

[I 2026-09-22 10:31:43,798] Trial 0 finished with value: 0.6216216286023458 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.6216216286023458.
[I 2026-09-22 10:31:52,230] Trial 1 finished with value: 0.6576576630274454 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 1 with value: 0.6576576630274454.
[I 2026-09-22 10:31:54,818] Trial 2 finished with value: 0.684684693813324 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 2 with value: 0.684684693813324.
[I 2026-09-22 10:31:57,879] Trial 3 finished with value: 0.6216216484705607 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.684684

[I 2026-09-22 10:33:49,072] A new study created in memory with name: no-name-59f96b19-919f-469f-99aa-4d3c1e028404


GCNII: 0.6162 +/- 0.0733

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-22 10:33:50,201] Trial 0 finished with value: 0.8828828930854797 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.8828828930854797.
[I 2026-09-22 10:33:52,672] Trial 1 finished with value: 0.8738738894462585 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.8828828930854797.
[I 2026-09-22 10:33:53,814] Trial 2 finished with value: 0.8738738894462585 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.8828828930854797.
[I 2026-09-22 10:33:55,591] Trial 3 finished with value: 0.8828829129536947 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 3 with value: 0.882882912953

In [3]:
! pip install torch_geometric
! pip install optuna
import copy
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import optuna
from optuna.samplers import TPESampler

from torch_geometric.datasets import WebKB
from torch_geometric.nn import GCNConv, TAGConv, SAGEConv, GATConv, APPNP, GCN2Conv, MessagePassing
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.transforms import RandomNodeSplit


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_data(dataset_name, split_type, seed):
    set_seed(seed)
    root = f"/tmp/{dataset_name}_{split_type}"
    if split_type == "fixed":
        dataset = WebKB(root=root, name=dataset_name)
    elif split_type == "60:20:20":
        transform = RandomNodeSplit(split="train_rest", num_val=0.2, num_test=0.2)
        dataset = WebKB(root=root, name=dataset_name, transform=transform)
    else:
        raise ValueError(f"Unknown split_type: {split_type}")
    return dataset, dataset[0]


class GCNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([GCNConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class TAGNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([TAGConv(dims[i], dims[i + 1], K=K) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class SAGENet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, dropout):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(num_layers)])
        self.dropout = dropout

    def forward(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)


class GATNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, heads, dropout):
        super().__init__()
        self.conv1 = GATConv(num_features, hidden, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden * heads, num_classes, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


class APPNPNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = APPNP(K=K, alpha=alpha)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GPRProp(MessagePassing):
    def __init__(self, K, alpha, init="PPR", **kwargs):
        super().__init__(aggr="add", **kwargs)
        self.K = K
        self.alpha = alpha
        if init == "PPR":
            gammas = alpha * (1 - alpha) ** np.arange(K + 1)
            gammas[-1] = (1 - alpha) ** K
        elif init == "Random":
            bound = np.sqrt(3.0 / (K + 1))
            gammas = np.random.uniform(-bound, bound, K + 1)
            gammas = gammas / np.sum(np.abs(gammas))
        else:
            raise ValueError(f"Unknown GPR init: {init}")
        self.gamma = nn.Parameter(torch.tensor(gammas, dtype=torch.float32))

    def forward(self, x, edge_index):
        edge_index, norm = gcn_norm(
            edge_index, edge_weight=None, num_nodes=x.size(0),
            add_self_loops=True, dtype=x.dtype
        )
        hidden = x * self.gamma[0]
        for k in range(self.K):
            x = self.propagate(edge_index, x=x, norm=norm)
            hidden = hidden + self.gamma[k + 1] * x
        return hidden

    def message(self, x_j, norm):
        return norm.view(-1, 1) * x_j


class GPRGNNNet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, K, alpha, init="PPR"):
        super().__init__()
        self.lin1 = nn.Linear(num_features, hidden)
        self.lin2 = nn.Linear(hidden, num_classes)
        self.prop = GPRProp(K=K, alpha=alpha, init=init)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.lin1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin2(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.prop(x, edge_index)
        return F.log_softmax(x, dim=1)


class GCNIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, dropout, num_layers, alpha, theta):
        super().__init__()
        self.lin_in = nn.Linear(num_features, hidden)
        self.convs = nn.ModuleList([
            GCN2Conv(hidden, alpha=alpha, theta=theta, layer=i + 1, shared_weights=True)
            for i in range(num_layers)
        ])
        self.lin_out = nn.Linear(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = x_org = F.relu(self.lin_in(x))
        for conv in self.convs:
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = F.relu(conv(x, x_org, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin_out(x)
        return F.log_softmax(x, dim=1)


class TAGIILayer(nn.Module):
    def __init__(self, in_channels, out_channels, x_org_channels, K, alpha_init, alpha_param=None):
        super().__init__()
        self.tagconv = TAGConv(in_channels, out_channels, K=K, normalize=True)
        self.need_proj = (x_org_channels != out_channels)
        if self.need_proj:
            self.proj = nn.Linear(x_org_channels, out_channels, bias=False)
        self.alpha = alpha_param if alpha_param is not None else nn.Parameter(torch.tensor(float(alpha_init)))

    def forward(self, x, x_org, edge_index):
        tag_out = self.tagconv(x, edge_index)
        residual = self.proj(x_org) if self.need_proj else x_org
        alpha = torch.sigmoid(self.alpha)
        return (1.0 - alpha) * tag_out + alpha * residual


class TAGIINet(nn.Module):
    def __init__(self, num_features, num_classes, hidden, num_layers, K,
                 dropout, alpha_init=0.05, shared_alpha=False):
        super().__init__()
        dims = [num_features] + [hidden] * (num_layers - 1) + [num_classes]
        shared_param = nn.Parameter(torch.tensor(float(alpha_init))) if shared_alpha else None
        self.layers = nn.ModuleList([
            TAGIILayer(dims[i], dims[i + 1], x_org_channels=num_features, K=K,
                       alpha_init=alpha_init, alpha_param=shared_param)
            for i in range(num_layers)
        ])
        self.dropout = dropout

    def forward(self, x, edge_index):
        x_org = x
        for i, layer in enumerate(self.layers):
            x = F.dropout(x, p=self.dropout, training=self.training)
            x = layer(x, x_org, edge_index)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        return F.log_softmax(x, dim=1)

    def alphas(self):
        return [torch.sigmoid(l.alpha).item() for l in self.layers]


def build_model(name, num_features, num_classes, cfg):
    if name == "GCN":
        return GCNNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "TAG":
        return TAGNet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"], cfg["dropout"])
    if name == "SAGE":
        return SAGENet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["dropout"])
    if name == "GAT":
        return GATNet(num_features, num_classes, cfg["hidden"], cfg["heads"], cfg["dropout"])
    if name == "APPNP":
        return APPNPNet(num_features, num_classes, cfg["hidden"], cfg["dropout"], cfg["K"], cfg["alpha"])
    if name == "GPRGNN":
        return GPRGNNNet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                         cfg["K"], cfg["alpha"], cfg.get("init", "PPR"))
    if name == "GCNII":
        return GCNIINet(num_features, num_classes, cfg["hidden"], cfg["dropout"],
                        cfg["num_layers"], cfg["alpha"], cfg["theta"])
    if name == "TAGII":
        return TAGIINet(num_features, num_classes, cfg["hidden"], cfg.get("num_layers", 2), cfg["K"],
                        cfg["dropout"], cfg.get("alpha_init", 0.05), cfg.get("shared_alpha", False))
    raise ValueError(f"Unknown model: {name}")


def train_one_run(name, cfg, dataset, data, device, epochs=1000, patience=100):
    model = build_model(name, dataset.num_node_features, dataset.num_classes, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    best_val_acc = 0.0
    best_train_acc = 0.0
    best_test_acc = 0.0
    best_state = None
    patience_counter = 0

    for _ in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.nll_loss(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out_eval = model(data.x, data.edge_index)
            pred = out_eval.argmax(dim=1)
            train_acc = (pred[data.train_mask] == data.y[data.train_mask]).float().mean().item()
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
            test_acc = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_train_acc = train_acc
            best_test_acc = test_acc
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    extra = {}
    if name == "TAGII" and best_state is not None:
        model.load_state_dict(best_state)
        extra["alphas"] = model.alphas()

    return {
        "train_acc": best_train_acc,
        "val_acc": best_val_acc,
        "test_acc": best_test_acc,
        **extra
    }


def run_seeds(name, cfg, dataset_name, split_type, seeds, device, epochs=1000, patience=100):
    results = []
    for seed in seeds:
        dataset, data = get_data(dataset_name, split_type, seed)
        data = data.to(device)
        set_seed(seed)
        results.append(train_one_run(name, cfg, dataset, data, device, epochs, patience))
    return results


# Optuna search
def suggest_config(trial, name):

    if name == "GCN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "TAG":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "K": trial.suggest_categorical("K", [2, 3]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "SAGE":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": 2,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    if name == "GAT":
        return {
            "hidden": trial.suggest_categorical("hidden", [8, 16, 32]),
            "heads": trial.suggest_categorical("heads", [4, 8]),
            "dropout": trial.suggest_categorical("dropout", [0.4, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "APPNP":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": trial.suggest_categorical("K", [5, 10]),
            "alpha": trial.suggest_categorical("alpha", [0.05, 0.1, 0.2]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GPRGNN":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "K": 10,
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2, 0.5, 0.9]),
            "init": trial.suggest_categorical("init", ["PPR", "Random"]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "GCNII":
        return {
            "hidden": trial.suggest_categorical("hidden", [32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [8, 16, 32]),
            "alpha": trial.suggest_categorical("alpha", [0.1, 0.2]),
            "theta": trial.suggest_categorical("theta", [0.5, 1.0]),
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3]),
        }
    if name == "TAGII":
        return {
            "hidden": trial.suggest_categorical("hidden", [16, 32, 64]),
            "num_layers": trial.suggest_categorical("num_layers", [2, 4, 8]),
            "K": trial.suggest_categorical("K", [2, 3]),
            "alpha_init": trial.suggest_categorical("alpha_init", [0.05, 0.1, 0.2]),
            "shared_alpha": False,
            "dropout": trial.suggest_categorical("dropout", [0.3, 0.5, 0.6, 0.7]),
            "lr": trial.suggest_categorical("lr", [0.005, 0.01, 0.02]),
            "weight_decay": trial.suggest_categorical("weight_decay", [5e-4, 1e-3, 5e-3]),
        }
    raise ValueError(f"Unknown model: {name}")


def optuna_search(name, dataset_name, split_type, device,
                  n_trials=20, tuning_seeds=(0, 1, 2),
                  epochs=1000, patience=100):
    def objective(trial):
        cfg = suggest_config(trial, name)
        seed_results = run_seeds(
            name, cfg, dataset_name, split_type, tuning_seeds,
            device, epochs=epochs, patience=patience
        )
        mean_val = float(np.mean([r["val_acc"] for r in seed_results]))
        mean_test = float(np.mean([r["test_acc"] for r in seed_results]))
        trial.set_user_attr("mean_test_acc_fyi", mean_test)
        trial.set_user_attr("cfg", cfg)
        return mean_val

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_trial = study.best_trial
    best_cfg = best_trial.user_attrs["cfg"]
    best_val = best_trial.value

    return best_cfg, best_val, study



# Experiment settings

DATASET_NAME = "Wisconsin"
SPLIT_TYPE = "60:20:20"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

N_TRIALS = 30
TUNING_SEEDS = (0, 1, 2)
FINAL_SEEDS = list(range(10, 20))

TUNING_EPOCHS = 1000
TUNING_PATIENCE = 100
FINAL_EPOCHS = 1000
FINAL_PATIENCE = 100

MODELS = ["GCN", "TAG", "SAGE", "GAT", "APPNP", "GPRGNN", "GCNII", "TAGII"]


def main():
    all_results = {}

    for name in MODELS:
        print(f"\n{'=' * 70}")
        print(f"Optuna search for {name}")
        print('=' * 70)

        best_cfg, best_val, study = optuna_search(
            name, DATASET_NAME, SPLIT_TYPE, DEVICE,
            n_trials=N_TRIALS,
            tuning_seeds=TUNING_SEEDS,
            epochs=TUNING_EPOCHS,
            patience=TUNING_PATIENCE,
        )

        print(f"Best config for {name}: {best_cfg}")
        print(f"Mean val acc (tuning seeds): {best_val:.4f}")

        print(f"Running final evaluation on {len(FINAL_SEEDS)} seeds...")
        final_results = run_seeds(
            name, best_cfg, DATASET_NAME, SPLIT_TYPE, FINAL_SEEDS, DEVICE,
            epochs=FINAL_EPOCHS, patience=FINAL_PATIENCE
        )

        test_accs = [r["test_acc"] for r in final_results]
        mean_acc = float(np.mean(test_accs))
        std_acc = float(np.std(test_accs))
        print(f"{name}: {mean_acc:.4f} +/- {std_acc:.4f}")

        all_results[name] = {
            "best_cfg": best_cfg,
            "tuning_mean_val_acc": best_val,
            "test_accs": test_accs,
            "test_acc_mean": mean_acc,
            "test_acc_std": std_acc,
            "n_trials": N_TRIALS,
            "best_trial_number": study.best_trial.number,
        }
        if name == "TAGII":
            all_results[name]["final_alphas_per_seed"] = [r.get("alphas") for r in final_results]

    print(f"\n{'#' * 70}")
    print(f"Summary - {DATASET_NAME} ({SPLIT_TYPE} split)")
    print('#' * 70)
    for name in MODELS:
        r = all_results[name]
        print(f"{name:8s}: {r['test_acc_mean']:.4f} +/- {r['test_acc_std']:.4f}   cfg={r['best_cfg']}")

    out_path = f"results_{DATASET_NAME}_{SPLIT_TYPE}_optuna.json"
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nSaved -> {out_path}")


if __name__ == "__main__":
    main()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


[I 2026-09-22 10:35:49,708] A new study created in memory with name: no-name-17fad219-ce13-47e2-8d8d-a7238cc8c2c5



Optuna search for GCN


  0%|          | 0/30 [00:00<?, ?it/s]

Processing...
Done!


...

[I 2026-09-22 10:39:59,294] Trial 0 finished with value: 0.6266666650772095 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.001}. Best is trial 0 with value: 0.6266666650772095.
[I 2026-09-22 10:40:05,290] Trial 1 finished with value: 0.6199999849001566 and parameters: {'hidden': 64, 'num_layers': 32, 'alpha': 0.2, 'theta': 1.0, 'dropout': 0.3, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 0 with value: 0.6266666650772095.
[I 2026-09-22 10:40:07,766] Trial 2 finished with value: 0.6799999872843424 and parameters: {'hidden': 64, 'num_layers': 8, 'alpha': 0.2, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.01, 'weight_decay': 0.001}. Best is trial 2 with value: 0.6799999872843424.
[I 2026-09-22 10:40:10,193] Trial 3 finished with value: 0.6133333047231039 and parameters: {'hidden': 32, 'num_layers': 8, 'alpha': 0.1, 'theta': 0.5, 'dropout': 0.6, 'lr': 0.005, 'weight_decay': 0.0005}. Best is trial 2 with value: 0.6799

[I 2026-09-22 10:42:56,057] A new study created in memory with name: no-name-eefa2424-8d81-413a-855f-ac11ceb3cf56


GCNII: 0.7760 +/- 0.0550

Optuna search for TAGII


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-22 10:42:57,363] Trial 0 finished with value: 0.8399999737739563 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.3, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.8399999737739563.
[I 2026-09-22 10:42:59,326] Trial 1 finished with value: 0.8399999737739563 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 3, 'alpha_init': 0.1, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 0 with value: 0.8399999737739563.
[I 2026-09-22 10:43:00,240] Trial 2 finished with value: 0.8466666340827942 and parameters: {'hidden': 32, 'num_layers': 2, 'K': 2, 'alpha_init': 0.05, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.005}. Best is trial 2 with value: 0.8466666340827942.
[I 2026-09-22 10:43:02,157] Trial 3 finished with value: 0.853333314259847 and parameters: {'hidden': 64, 'num_layers': 4, 'K': 2, 'alpha_init': 0.2, 'dropout': 0.6, 'lr': 0.02, 'weight_decay': 0.0005}. Best is trial 3 with value: 0.8533333142598